### Introduction

In this tutorial, we run a low-energy PKA cascade simulation to explore defect generation through atomic displacements. Starting from a basic input with a constant timestep, we progressively incorporate recently implemented modules in the TurboGAP code that are relevant to cascade dynamics. Specifically, we examine **adaptive timestep calculation** and the inclusion of **electronic stopping** to more accurately capture the high-energy dynamics of the cascade.


### Implementation details

In this tutorial, we implement the TDE calculation through a Python workflow. 

The implementation follows three main phases:

* **Phase 1: Cell Equilibration** We begin by thermalizing Silicon supercell at a target temperature. This ensures the lattice contains realistic thermal displacements and velocities before the impact event occurs.
    
* **Phase 2: PKA Selection and Velocity Initialization** A specific atom is designated as the **PKA**. Based on the chosen crystallographic direction, we calculate the velocity components $(v_x, v_y, v_z)$ required to match the target PKA energy in electronvolts (eV).
    
* **Phase 3: MD Execution and Defect Analysis** The dynamics are launched. After the simulation, we visually check the trajectory to determine if a stable defect has been created. If no defect is found, the simulation should be run with the $E+\Delta E$ PKA energy.


In [ ]:
from weas_widget import WeasWidget
from ase.build import bulk
from ase.io import write, read
import numpy as np
import math
import subprocess
import matplotlib.pyplot as plt

In [ ]:
# The following functions will be used throughout the notebook.
# The first visualizes structures and trajectories, and the
# second runs the TurboGAP code.

def visualize(atoms):
    """
    Visualize the given structure or trajectory.

    Parameters:
        ASE's atoms object

    Returns:
        'viewer', the visual handle of the given object
    """
    viewer = WeasWidget()
    viewer.avr.model_style = 0 # ball mode
    viewer.from_ase(atoms)
    viewer.avr.species.settings["Si"].update({"radius": 0.5})
    return viewer


def run_turbogap(n_cores=4, capture_output=False):
    """
    Run the TurboGAP code in 'md' mode (turbogap md).
    The 'input' file and 'gap_files/' directory should be present
    in the working directory. 
    """
    command = ['mpirun', '-np', str(n_cores), 'turbogap_stopping', 'md']
    print("Running TurboGAP simulation... \n")
    result = subprocess.run(command, capture_output=capture_output, check=True)
    print("Simulation complete!")

In [ ]:
# Creating the pristine cell:

ao = 5.467 #the optimum lattice constant predicted by the poential
element = 'Si'
lattice = 'diamond'
rx, ry, rz = 3, 3, 3 

atoms_pristine = bulk(element, lattice, ao, cubic=True)
atoms_pristine = atoms_pristine.repeat([rx, ry, rz])

pcell = 'pristine.xyz'
write(pcell, atoms_pristine)

viewer = visualize(atoms_pristine); viewer

In [ ]:
# Thermalizing the cell at the given temperature. Thermalization is done to
# make the simulation results comparable with experimental values measured at
# the same temperature. Also, the random motion of atoms creates a more realistic
# collision probability between the lattice atoms and the atom in motion.

def create_tgap_input_eq(cell_name, temp, md_nsteps):
    """
    Creates TurboGAP input file to run NPT.

    Parameters:
        cell_name : structure file in xyz format
        temp      : temperature
        md_nsteps : number of steps
    
    Returns:
        the 'input' file is written in the directory
    """
    input_str = f"""! Species-specific info
    atoms_file = '{cell_name}'
    pot_file = 'gap_files/Si.ZBL-stiff.TGAP.gap'
    n_species = 1
    species = Si
    masses = 28.09   
    
    md_nsteps = {md_nsteps}
    md_step = 1.
    
    t_beg = {temp}
    t_end = {temp}
    tau_t = 100.
    thermostat = bussi 
    
    p_beg = 1
    p_end = 1
    tau_p = 200.
    gamma_p = 40 !B_Si/B_water
    barostat = berendsen
    barostat_sym = diag
    
    write_xyz = 200
    write_thermo = 1
    write_lv = .true.
    """
    f = open('input', 'w')
    f.write(input_str)
    f.close()


#---------- Controller variables 1 -----------------#
# Number of MD steps for equilibrating the pristine cell:
md_nsteps_eq = 3000

# Equilibration temperature (K) of the pristine cell:
temp_eq = 30
#------------------------------------------------------

# Now, let's run cell equilibration:
create_tgap_input_eq(pcell, temp_eq, md_nsteps_eq)
run_turbogap()


In [ ]:
# Check how the temperature and pressure vary:
steps, T, epot, P = np.loadtxt('thermo.log', usecols=(0,2,4,5), unpack=True)
fig, axs = plt.subplots(1, 3, figsize=(12,4), sharex=True)
axs[0].plot(steps, T)
axs[0].set_ylabel("Temperature (K)")
axs[1].plot(steps, P)
axs[1].set_ylabel("Pressure (bar)")
axs[2].plot(steps, epot)
axs[2].set_ylabel("Potential energy (eV)")
fig.supxlabel("Steps")
plt.tight_layout()
plt.show()


In [ ]:
# The last frame of the equilibrated trajectory is taken to perform TDE calculation:
atoms_eq = read('trajectory_out.xyz', index='-1')

viewer = visualize(atoms_eq); viewer


In [ ]:
# to run the TDE simulation, the following function will be used.

def calc_velocity(E, direction, element):
    """
    Calculates velocity components (Ang/ps) to be assigned to the PKA atom

    Parameters:
        E:          kinetic energy of PKA (eV)
        direction:  [h,k,l]
        element:    symbol of the element (e.g 'Si')

    Returns:
        Velocity components: (Vx, Vy, Vz) in [Ang/ps]
    
    """
    m = {'Si':28.0855, 'C':12.011, 'Ge':72.64}
    #
    Vtot = np.sqrt((2.0 * 1.60218*10**8 * E) / (m[element] * 1.66054)) / 100.0
    # 
    vec = np.asarray(direction, dtype=float)
    norm = np.linalg.norm(vec)
    unit_vec = vec / norm
    #
    Vx = Vtot * unit_vec[0] 
    Vy = Vtot * unit_vec[1]
    Vz = Vtot * unit_vec[2]
    #
    return Vx, Vy, Vz


#---------- Controller variables 2 -----------------#
# TDE dir:
tde_dir = [2, 4, 3]

# The energy of the PKA atom (eV):
E_pka = 20

# The index of the PKA atom in the pristine cell:
PKA_indx = 104
#------------------------------------------------------

# Velocity components:
Vx, Vy, Vz = calc_velocity(E_pka, tde_dir, 'Si') 

# Setting the velocity vector for the PKA atom here.
atoms_pka = atoms_eq.copy()
atoms_pka.arrays['velocities'][PKA_indx] = np.array([Vx, Vy, Vz]) / 1000 #Ang/fs

# And exporting the cell to be read by TurboGAP:
cell_tde = 'tde.xyz'
write(cell_tde, atoms_pka)


In [ ]:
# Here we initiate the PKA and examine the resulting defect generation. 
# We run the cascade in "four" different inputs or "modes".
# In each mode, a new module is added to the input.

# For each mode, we update the input to activate the corresponding module
# and then run the cascade. In every case, we start with E_PKA = 10 eV
# and gradually increase it until a stable defect is formed.

# We aim to assess how the inclusion of different modules affects defect
# formation. Note that, since the initial PKA energy is relatively low,
# the final outcomes may not differ significantly. Nevertheless, the goal
# is to learn how to run cascade simulations using these features.

# Each mode and its corresponding input are described below:

#----------------------------------------------------------------------------------
# Mode 1: A "constant" timestep  |  no adaptive timestep  |  no electronic stopping
#----------------------------------------------------------------------------------
# 
# To initiate this mode, specify a constant timestep in the 
# "create_tgap_input_tde" function using the following command:
# 
#   md_step = 2 !fs
#
# Start the cascade with E_pka = 10 eV and increase it until you create 
# a stable defect. Store the value for each mode to compare with the 
# values obtained from the other modes.
#
#
#-----------------------------------------------------------------------------------
# Mode 2: adaptive timestep  |  no electronic stopping
#-----------------------------------------------------------------------------------
#
# To initiate this mode, remove the "md_step" command and add
# the following to the create_tgap_input_tde function:
# 
#   adaptive_time = .true.
#    adapt_time_groupID = all
#    adapt_tstep_interval = 1
#    adapt_tmin = 1.0E-10
#    adapt_tmax = 1.0E-00
#    adapt_xmax = 1.0E-01
#
# Start the cascade with E_pka = 10 eV and increase it until you create 
# a stable defect. Store the value for each mode to compare with the 
# values obtained from the other modes.
#
#
#-----------------------------------------------------------------------------------
# Mode 3: adaptive timestep  |  friction-based electronic stopping model (FES)
#-----------------------------------------------------------------------------------
#
# In this mode, in addition to adaptive timestep calculation, we include
# electronic stopping in the simulation. First, we use the traditional
# FES model, which is activated with the following commands:
# 
#   electronic_stopping = .true.
#    eel_groupID = all
#    eel_cut = 1.0
#    eel_freq_out = 10
#    estop_filename = 'Si-in-Si-el-stopping.txt'
# 
# Please copy the "Si-in-Si-el-stopping.txt" file from the provided files
# before running TurboGAP. This file is found within "files" directory.
#
# Start the cascade with E_pka = 10 eV and increase it until you create 
# a stable defect. Store the value for each mode to compare with the
# values obtained from the other modes.
#
#
#-----------------------------------------------------------------------------------
# Mode 4: adaptive timestep  |  unified two-temperature electronic stopping model (UTTM)
#-----------------------------------------------------------------------------------
#
# In the last mode, we keep the adaptive timestep calculation, and include the UTTM 
# stopping model instead of the FES model. Please remove the commands for the FES model
# and replace them with the following commands in order to activate the UTTM model:
#
#   nonadiabatic_processes = .true.
#    eph_groupID = all
#    eph_fdm_option = 1
#    eph_friction_option = 1
#    eph_random_option = 1
#    eph_betafile = 'beta.dat'
#    eph_box_limits = 0.0 16.4 0.0 16.4 0.0 16.4
#    eph_Ti_e = 30.0
#    eph_rho_e = 1.0
#    eph_C_e = 9.738E-05
#    eph_kappa_e = 9.738E-02
#    eph_gsx = 1
#    eph_gsy = 1
#    eph_gsz = 1
#
# Please copy the "beta.dat" file from the provided files before running TurboGAP. 
# Note that this "beta file" is a simplified model created for illustrative
# purposes in this tutorial. It was generated by Rafael Nuñez Palacio in the
# Department of Applied Physics at Aalto University.
#
# Start the cascade with E_pka = 10 eV and increase it until you create 
# a stable defect. Store the value for each mode to compare with the
# values obtained from the other modes.

def create_tgap_input_tde(cell_name, md_nsteps):
    """
    Creates TurboGAP input file to run TDE simulation.

    Parameters:
        cell_name : structure file in xyz format
        md_nsteps : number of steps
    
    Returns:
        The 'input' file is written in the directory

    """
    input_str = f"""! Species-specific info
    atoms_file = '{cell_name}'
    pot_file = 'gap_files/Si.ZBL-stiff.TGAP.gap'
    n_species = 1
    species = Si
    masses = 28.09   

    md_nsteps = {md_nsteps}

    
    ! Introduce the changes here:
    adaptive_time = .true.
    adapt_time_groupID = all
    adapt_tstep_interval = 1
    adapt_tmin = 1.0E-10
    adapt_tmax = 1.0E-00
    adapt_xmax = 1.0E-01
    
    
   nonadiabatic_processes = .true.
    eph_groupID = all
    eph_fdm_option = 1
    eph_friction_option = 1
    eph_random_option = 1
    eph_betafile = 'beta.dat'
    eph_box_limits = 0.0 16.4 0.0 16.4 0.0 16.4
    eph_Ti_e = 30.0
    eph_rho_e = 1.0
    eph_C_e = 9.738E-05
    eph_kappa_e = 9.738E-02
    eph_gsx = 1
    eph_gsy = 1
    eph_gsz = 1    
    
    
    
    write_thermo = 1
    write_xyz = 10
    """
    f = open('input', 'w')
    f.write(input_str)
    f.close()


#---------- Controller variables 4 -----------------#
# Number of MD steps in TDE simulations:
md_nsteps_tde = 1000
#------------------------------------------------------

create_tgap_input_tde(cell_tde, md_nsteps_tde)
run_turbogap(capture_output=False)


In [ ]:
# Let's visually check if the defect was generated. If not, 
# we should increase the PKA energy and with an specific 
# interval, 1 or 2 eV?

# Please continue increasing the energy, until you find a
# stable defect!

traj = read('trajectory_out.xyz', index=':')
viewer = visualize(traj); viewer